# Sparky custom Piper voice

Two parts, run on **free Colab with a T4 GPU** (Runtime > Change runtime type > T4 GPU).
Everything is saved in Google Drive under `MyDrive/sparky_voice/`, so you can stop and resume any time.

**Before you start**, upload these two files to `MyDrive/sparky_voice/`:
- `sentences.txt` (from the repo's `voice_training/` folder)
- `reference.wav` (your 20-30 s Sparky-voice clip)

**Part A** generates the training clips with Chatterbox (MIT licence) cloning your reference clip.
**Part B** fine-tunes a Piper voice on those clips.

Between Part A and Part B: **Runtime > Disconnect and delete runtime**, then reconnect.
The two parts need different library versions and will clash in the same runtime.

## Part A - generate training clips (~1-2 h)

In [ ]:
# A1. Mount Drive and check the GPU
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
ROOT = '/content/drive/MyDrive/sparky_voice'
for f in ['sentences.txt', 'reference.wav']:
    assert os.path.exists(f'{ROOT}/{f}'), f'Upload {f} to {ROOT} first'
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
      or 'NO GPU - Runtime > Change runtime type > T4 GPU')

In [ ]:
# A2. Install Chatterbox + Whisper. The "dependency resolver" ERROR lines about torchvision/starlette are harmless.
# Afterwards: Runtime > Restart session (Chatterbox installs an older torch), re-run A1, then continue at A3.
!pip install -q chatterbox-tts faster-whisper num2words
!pip uninstall -q -y torchvision  # Colab's torchvision needs the newer torch and can crash imports; Chatterbox doesn't use it

In [ ]:
# A3. Generate clips. Safe to re-run: clips already on Drive are skipped.
import re, difflib, torchaudio as ta
from chatterbox.tts import ChatterboxTTS
from faster_whisper import WhisperModel
from num2words import num2words

LIMIT = None      # set to 10 for a quick smoke test first
MIN_MATCH = 0.85  # how closely Whisper's transcript must match the sentence to keep a clip
TAKES = [dict(exaggeration=0.5, cfg_weight=0.5),   # natural
         dict(exaggeration=0.7, cfg_weight=0.4)]   # more excited

WAVS = f'{ROOT}/wavs'
os.makedirs(WAVS, exist_ok=True)
sentences = [s.strip() for s in open(f'{ROOT}/sentences.txt', encoding='utf-8') if s.strip()][:LIMIT]
tts = ChatterboxTTS.from_pretrained(device='cuda')
asr = WhisperModel('small', device='cuda', compute_type='float16')

def words(t):
    t = re.sub(r'(?<=\d),(?=\d{3})', '', t.lower()).replace(' km', ' kilometres')  # "9,000 km"
    t = re.sub(r'\d+', lambda m: num2words(int(m.group())), t)  # Whisper writes "42", we wrote "forty two"
    return re.sub(r'[^a-z]+', ' ', t).split()

rows, rejected = [], []
for i, text in enumerate(sentences):
    for t, params in enumerate(TAKES):
        name = f'{i:04d}_{t}.wav'
        path = f'{WAVS}/{name}'
        if not os.path.exists(path):
            for attempt in range(3):
                wav = ta.functional.resample(
                    tts.generate(text, audio_prompt_path=f'{ROOT}/reference.wav', **params), tts.sr, 22050)
                # pass audio straight to Whisper as a 16 kHz array - its file loader breaks with Chatterbox's PyAV version
                audio16k = ta.functional.resample(wav, 22050, 16000)[0].cpu().numpy()
                heard = ' '.join(s.text for s in asr.transcribe(audio16k, language='en')[0])
                if difflib.SequenceMatcher(None, words(text), words(heard)).ratio() >= MIN_MATCH:
                    ta.save(path, wav, 22050, encoding='PCM_S', bits_per_sample=16)
                    break
            else:
                rejected.append((text, heard))
                continue
        rows.append(f'{name}|{text}')
    if i % 25 == 0:
        print(f'{i}/{len(sentences)} sentences | {len(rows)} clips kept | {len(rejected)} rejected')

with open(f'{ROOT}/metadata.csv', 'w', encoding='utf-8') as f:
    f.write('\n'.join(rows) + '\n')
print(f'\nDone: {len(rows)} clips kept, {len(rejected)} rejected')
for text, heard in rejected[:20]:
    print(f'  REJECTED: {text!r}\n     heard: {heard!r}')

In [ ]:
# A4. Listen to a few random clips - they should sound like you, clearly saying the sentence
import random
from IPython.display import Audio, display
for r in random.sample(rows, min(4, len(rows))):
    f, t = r.split('|', 1)
    print(t)
    display(Audio(f'{WAVS}/{f}'))

## Part B - fine-tune Piper (several hours, resumable)

**First: Runtime > Disconnect and delete runtime**, reconnect with a T4 GPU, then run B1-B4.

Colab free sessions time out after a few hours. When that happens, just run B1-B4 again:
training resumes from `latest.ckpt` on Drive (backed up every 15 minutes).

To check how it sounds mid-training: press **Stop** on B4, run **B5**, then re-run B4 to continue.

In [ ]:
# B1. Mount Drive and check the GPU
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
ROOT = '/content/drive/MyDrive/sparky_voice'
assert os.path.exists(f'{ROOT}/metadata.csv'), 'Run Part A first'

# Old Piper checkpoints store Python objects that newer torch refuses to load by default, and
# Lightning requests the strict mode explicitly, so patch torch.load in the training subprocesses.
# Only ever loads the official rhasspy checkpoint and our own backups - trusted files.
os.makedirs('/content/torchfix', exist_ok=True)
with open('/content/torchfix/sitecustomize.py', 'w') as f:
    f.write("import torch\n_load = torch.load\n"
            "def load(*a, **k):\n    k['weights_only'] = False\n    return _load(*a, **k)\n"
            "torch.load = load\n")
os.environ['PYTHONPATH'] = '/content/torchfix:' + os.environ.get('PYTHONPATH', '')
print(sum(1 for _ in open(f'{ROOT}/metadata.csv')), 'training clips')
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
      or 'NO GPU - Runtime > Change runtime type > T4 GPU')

In [ ]:
# B2. Install piper1-gpl training code (~5 min)
!apt-get -qq install -y espeak-ng build-essential cmake ninja-build > /dev/null
!git clone -q https://github.com/OHF-voice/piper1-gpl.git /content/piper1-gpl
%cd /content/piper1-gpl
!pip install -q -e '.[train]' scikit-build onnxscript
# newer torch defaults to the dynamo ONNX exporter, which chokes on VITS - use the classic exporter
!sed -i 's/torch.onnx.export(/torch.onnx.export(dynamo=False, /' src/piper/train/export_onnx.py
!./build_monotonic_align.sh
!python setup.py build_ext --inplace

In [ ]:
# B3. Download the base voice to fine-tune from (LJSpeech medium - public-domain dataset)
from huggingface_hub import hf_hub_download
import sys, inspect, torch
SRC = hf_hub_download('rhasspy/piper-checkpoints', 'en/en_US/ljspeech/medium/lj-med_1000.ckpt',
                      repo_type='dataset', local_dir='/content/base')

# The old checkpoint stores old-Piper trainer settings (e.g. sample_bytes) that piper1-gpl's CLI
# rejects ("fit does not accept option model.sample_bytes") - keep only VitsModel's own settings.
sys.path.insert(0, '/content/piper1-gpl/src')  # editable install isn't importable in this kernel until restart
from piper.train.vits.lightning import VitsModel
ckpt = torch.load(SRC, map_location='cpu', weights_only=False)
accepted = set(inspect.signature(VitsModel.__init__).parameters) - {'self', 'batch_size', 'num_symbols'}
ckpt['hyper_parameters'] = {k: v for k, v in ckpt.get('hyper_parameters', {}).items() if k in accepted}
BASE = '/content/base/lj-med_1000_clean.ckpt'
torch.save(ckpt, BASE)
print(BASE)

In [ ]:
# B4. Train in the background; this cell watches it every minute and backs up to Drive every 15 min.
# Pressing Stop on this cell only stops WATCHING - training keeps going. Re-run B4 to watch again.
# To stop training for real, run:  proc.terminate(); backup()
import glob, re, shutil, subprocess, time
MAX_EPOCHS = 1300     # base checkpoint is at epoch 1000, so this is ~300 epochs of fine-tuning (~6-7 h on a T4)
VAL_EVERY = 5         # quality check + checkpoint every N epochs (each check costs ~1.5 min)
BATCH_SIZE = 16       # lower to 8 if you get CUDA out-of-memory
SAVE_EVERY_MIN = 15
DRIVE_CKPT = f'{ROOT}/latest.ckpt'
LOG = '/content/train.log'

def backup():
    ckpts = glob.glob('/content/train/**/*.ckpt', recursive=True)
    if not ckpts:
        print('--- no checkpoint to back up yet (the first one is written after the first epoch)', flush=True)
        return
    newest = max(ckpts, key=os.path.getmtime)
    shutil.copy(newest, DRIVE_CKPT + '.tmp')
    os.replace(DRIVE_CKPT + '.tmp', DRIVE_CKPT)
    print(f'--- backed up {os.path.basename(newest)} to Drive', flush=True)

def log_tail(n):
    text = open(LOG, errors='replace').read().replace('\r', '\n')
    return [l for l in text.split('\n') if l.strip()][-n:]

def progress():
    # the training log has no usable progress bar here, so read progress from checkpoint file names
    names = [os.path.basename(f) for f in glob.glob('/content/train/**/*.ckpt', recursive=True)]
    epochs = [int(m.group(1)) for n in names if (m := re.match(r'epoch=(\d+)', n))]
    mos = [float(m.group(1)) for n in names if (m := re.search(r'val_mos=(\d+\.\d+)', n))]
    if not epochs:
        return 'warming up (first checkpoint not written yet)'
    return f'checkpoint at epoch {max(epochs)} of {MAX_EPOCHS - 1}' + (f' | best voice quality (val_mos) {max(mos):.2f}' if mos else '')

if 'proc' in globals() and proc.poll() is None:
    print('Training is already running - watching it.')
else:
    start = DRIVE_CKPT if os.path.exists(DRIVE_CKPT) else BASE
    print('Starting from', start)
    cmd = ['python', '-m', 'piper.train', 'fit',
           '--data.voice_name', 'sparky',
           '--data.csv_path', f'{ROOT}/metadata.csv',
           '--data.audio_dir', f'{ROOT}/wavs',
           '--model.sample_rate', '22050',
           '--data.espeak_voice', 'en-gb-x-rp',  # British pronunciation rules - matches the speaker's accent
           '--data.cache_dir', '/content/cache',
           '--data.config_path', f'{ROOT}/sparky.onnx.json',
           '--data.batch_size', str(BATCH_SIZE),
           '--trainer.max_epochs', str(MAX_EPOCHS),
           '--trainer.check_val_every_n_epoch', str(VAL_EVERY),
           '--trainer.default_root_dir', '/content/train',
           '--ckpt_path', start]
    # start_new_session: Colab's Stop button interrupts the whole process group - keep training out of it
    proc = subprocess.Popen(cmd, stdout=open(LOG, 'w'), stderr=subprocess.STDOUT, cwd='/content/piper1-gpl',
                            env={**os.environ, 'PYTHONUNBUFFERED': '1'}, start_new_session=True)

last_save = time.time()
try:
    while proc.poll() is None:
        time.sleep(60)
        gpu = subprocess.run(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used', '--format=csv,noheader'],
                             capture_output=True, text=True).stdout.strip()
        print(f'{time.strftime("%H:%M")} | GPU {gpu} | {progress()}', flush=True)
        if time.time() - last_save > SAVE_EVERY_MIN * 60:
            backup()
            last_save = time.time()
    print('\n'.join(log_tail(30)))
    print('Training process ended, exit code', proc.returncode)
    backup()
except KeyboardInterrupt:
    print('Stopped watching - training is still running. Run B5 to listen, or re-run B4 to keep watching.')

In [ ]:
# B5. Export the latest checkpoint to a Piper voice and listen to it
import shutil, wave
from IPython.display import Audio, display
backup()  # grab the newest checkpoint first (training can keep running while you listen)
OUT = f'{ROOT}/en_US-sparky-medium.onnx'
if os.path.exists(OUT):
    os.remove(OUT)  # so a failed export can't silently leave an old voice behind
!python -m piper.train.export_onnx --checkpoint {DRIVE_CKPT} --output-file {OUT}
assert os.path.exists(OUT), 'Export failed - see the error above'
shutil.copy(f'{ROOT}/sparky.onnx.json', OUT + '.json')

from piper import PiperVoice
voice = PiperVoice.load(OUT)
with wave.open('/content/test.wav', 'wb') as w:
    voice.synthesize_wav("Hi there! I'm Sparky, and I love questions! "
                         "Did you know that octopuses have three hearts? Isn't that amazing?", w)
display(Audio('/content/test.wav'))
print(f'Voice files are in Drive: {OUT} and {OUT}.json')

## When you're happy with the voice

Download `en_US-sparky-medium.onnx` and `en_US-sparky-medium.onnx.json` from `MyDrive/sparky_voice/`
into the repo's `data/piper_voices/`, then start Sparky with:

```
PIPER_VOICE=en_US-sparky-medium python simple_api.py
```